### Preparación del Entorno

In [0]:
# Cargamos la tabla limpia y la convertimos en una Vista Temporal de Spark
df_citas = spark.table("default.citas_pmm_limpioV3")
df_citas.createOrReplaceTempView("v_citas_pmm")

print("Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.")

Vista 'v_citas_pmm' creada exitosamente. Lista para análisis SQL.


### N1: Rendimiento Financiero y Distribución de Cobertura por Aseguradora

In [0]:
%sql
CREATE OR REPLACE TABLE rendimiento_aseguradora AS
SELECT 
    nom_aseguradora AS Aseguradora,
    COUNT(*) AS Volumen_Citas,
    ROUND(SUM(pago_aseg), 2) AS Total_Pagado_Aseguradora_USD,
    ROUND(SUM(pago_clie), 2) AS Total_Copago_Paciente_USD,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND((SUM(pago_aseg) / SUM(pago_total)) * 100, 2) AS Porcentaje_Cobertura_Promedio
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY nom_aseguradora
ORDER BY Ingresos_Totales_USD DESC;

SELECT * FROM rendimiento_aseguradora;

Aseguradora,Volumen_Citas,Total_Pagado_Aseguradora_USD,Total_Copago_Paciente_USD,Ingresos_Totales_USD,Porcentaje_Cobertura_Promedio
Particular / Sin Seguro,2016,0.0,150787.35,150787.35,0.0
Seguros ASSA,1272,66379.81,28909.3,95289.11,69.66
PALIG,1246,65566.63,27427.87,92994.5,70.51
MAPFRE Panamá,1238,64433.25,27404.52,91837.77,70.16
Seguros SURA,1206,61975.38,26534.68,88510.06,70.02
Compañía Internacional de Seguros (IS),1086,56860.12,23654.76,80514.88,70.62


### N2: Tasa de Cancelación por Sucursal

In [0]:
%sql
CREATE OR REPLACE TABLE cancelacion_sucursal AS
SELECT 
    nom_sucursal AS Sucursal,
    COUNT(*) AS Citas_Agendadas,
    SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) AS Citas_Canceladas,
    SUM(CASE WHEN estado_cita = 'Completada' THEN 1 ELSE 0 END) AS Citas_Completadas,
    ROUND((SUM(CASE WHEN estado_cita = 'Cancelada' THEN 1 ELSE 0 END) / COUNT(*)) * 100, 2) AS Tasa_Cancelacion_PCT
FROM v_citas_pmm
GROUP BY nom_sucursal
ORDER BY Tasa_Cancelacion_PCT DESC;
SELECT * FROM cancelacion_sucursal;

Sucursal,Citas_Agendadas,Citas_Canceladas,Citas_Completadas,Tasa_Cancelacion_PCT
PMM El Dorado,2019,404,1615,20.01
PMM Brisas del Golf,1452,289,1163,19.9
PMM Costa del Este,2521,489,2032,19.4
PMM San Francisco,2990,565,2425,18.9
PMM Tocumen,1018,189,829,18.57


### N3: Segmentación Demográfica y Ticket Promedio

In [0]:
%sql
CREATE OR REPLACE TABLE segmentacion_demografica AS
SELECT 
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END AS Segmento_Poblacional,
    COUNT(*) AS Total_Atenciones,
    ROUND(SUM(pago_total), 2) AS Facturacion_Total_USD,
    ROUND(AVG(pago_total), 2) AS Ticket_Promedio_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY 
    CASE 
        WHEN edad_pac_cita < 15 THEN '1. Pediátrico (<15 años)'
        WHEN edad_pac_cita BETWEEN 15 AND 64 THEN '2. Adulto (15-64 años)'
        ELSE '3. Geriátrico (65+ años)' 
    END
ORDER BY Segmento_Poblacional ASC;
SELECT * FROM segmentacion_demografica;

Segmento_Poblacional,Total_Atenciones,Facturacion_Total_USD,Ticket_Promedio_USD
1. Pediátrico (<15 años),1671,109453.23,65.5
2. Adulto (15-64 años),4675,349552.84,74.77
3. Geriátrico (65+ años),1718,140927.6,82.03


### N4: Rentabilidad de cada Especialidad por Hora

In [0]:
%sql
CREATE OR REPLACE TABLE rentabilidad_especialidad AS
SELECT 
    especialidad_medica AS Especialidad,
    COUNT(*) AS Total_Consultas,
    ROUND(SUM(mins_cit) / 60, 2) AS Horas_Totales_Invertidas,
    ROUND(SUM(pago_total), 2) AS Ingresos_Totales_USD,
    ROUND(SUM(pago_total) / (SUM(mins_cit) / 60), 2) AS Rentabilidad_Por_Hora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada'
GROUP BY especialidad_medica
ORDER BY Rentabilidad_Por_Hora_USD DESC;
SELECT * FROM rentabilidad_especialidad;

Especialidad,Total_Consultas,Horas_Totales_Invertidas,Ingresos_Totales_USD,Rentabilidad_Por_Hora_USD
Neurología,449,277.0,51348.84,185.37
Nefrología,425,258.25,43805.82,169.63
Cardiología,342,211.5,35647.89,168.55
Gastroenterología,428,280.5,42580.15,151.8
Psiquiatría,450,271.75,40048.95,147.37
Radiología,744,479.75,62437.0,130.14
Dermatología,723,447.25,54309.52,121.43
Geriatría,113,69.5,8417.34,121.11
Otorrinolaringología,751,469.25,56286.5,119.95
Oftalmología,743,456.75,50042.27,109.56


### N5: Análisis de Citas por Jornada y Hora

In [0]:
%sql
CREATE OR REPLACE TABLE citas_jornadas AS
SELECT 
    CASE 
        -- REGEXP_EXTRACT busca la primera coincidencia de hora (ej: "09:30") y extrae solo los primeros 2 dígitos (\d{2})
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END AS Jornada_Horaria,
    COUNT(*) AS Cantidad_Citas,
    ROUND((COUNT(*) / (SELECT COUNT(*) FROM v_citas_pmm)) * 100, 2) AS Porcentaje_Demanda_PCT
FROM v_citas_pmm
GROUP BY 
    CASE 
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) < 12 THEN 'Mañana (07:00 - 11:59)'
        WHEN CAST(REGEXP_EXTRACT(hr_inicio_cit, '(\\d{2}):\\d{2}', 1) AS INT) BETWEEN 12 AND 16 THEN 'Tarde (12:00 - 16:59)'
        ELSE 'Noche (17:00 - 20:00)'
    END
ORDER BY Cantidad_Citas DESC;
SELECT * FROM citas_jornadas;

Jornada_Horaria,Cantidad_Citas,Porcentaje_Demanda_PCT
Tarde (12:00 - 16:59),4553,45.53
Mañana (07:00 - 11:59),4454,44.54
Noche (17:00 - 20:00),993,9.93


### N6: Reembolso Promedio de Aseguradoras por Especialidad

In [0]:
%sql
CREATE OR REPLACE TABLE citas_especialidad AS
SELECT 
    especialidad_medica AS Especialidad,
    COUNT(*) AS Citas_Aseguradas,
    ROUND(SUM(pago_aseg), 2) AS Facturacion_A_Seguros_USD,
    ROUND(AVG(pago_aseg), 2) AS Reembolso_Promedio_Aseguradora_USD
FROM v_citas_pmm
WHERE estado_cita = 'Completada' AND nom_aseguradora != 'Sin Seguro'
GROUP BY especialidad_medica
ORDER BY Reembolso_Promedio_Aseguradora_USD DESC;
SELECT * FROM citas_especialidad;

Especialidad,Citas_Aseguradas,Facturacion_A_Seguros_USD,Reembolso_Promedio_Aseguradora_USD
Neurología,449,26451.39,58.91
Cardiología,342,18924.16,55.33
Nefrología,425,22640.87,53.27
Gastroenterología,428,21521.32,50.28
Psiquiatría,450,21215.68,47.15
Radiología,744,33485.62,45.01
Geriatría,113,4576.68,40.5
Otorrinolaringología,751,30273.34,40.31
Dermatología,723,28827.66,39.87
Ginecología,259,9664.22,37.31
